# LeetCode 3967 — Finish Time of Tasks II bằng Python

> **Bài Premium: đề bài viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.** Tên phương thức dưới đây (`minFinishTime`) được suy ra từ đề (không có skeleton công khai cho bài Premium) — cần đối chiếu lại với trình soạn thảo thật (tên có thể khác, ví dụ `finishTime`).

- Link: https://leetcode.com/problems/finish-time-of-tasks-ii/
- Độ khó: **Hard**
- Chủ đề: Cây, DFS, quy hoạch động trên cây, **đổi gốc (rerooting)**, min/max của các nhánh con

Skeleton (suy luận, cần đối chiếu):

```python
class Solution:
    def minFinishTime(self, n: int, edges: list[list[int]], baseTime: list[int]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Có `n` công việc nối thành một cây vô hướng. Chọn một công việc làm **gốc**; khi đó mỗi công việc có "thời điểm hoàn thành" tính từ lá lên: lá thì bằng `baseTime`, nút trong thì phụ thuộc vào thời điểm hoàn thành **sớm nhất** và **muộn nhất** của các con. Cần chọn gốc để thời điểm hoàn thành **của gốc** là nhỏ nhất.

Nếu thử từng gốc rồi chạy DFS thì `O(n^2)`, quá chậm với `n = 10^5`. Đây là bài **rerooting** (đổi gốc) kinh điển: tính một lần DP "xuống" từ gốc 0, rồi một lần DP "lên" để biết giá trị của **nhánh phía cha** khi nhìn từ mỗi nút. Vì công thức chỉ dùng `min` và `max` của các con, ta chỉ cần giữ **hai giá trị lớn nhất / hai giá trị nhỏ nhất** để "loại bỏ một con".

Sau notebook này bạn sẽ:

1. Đọc đúng định nghĩa đệ quy "finish time" và tự tính tay cho một gốc cố định.
2. Hiểu bài toán đổi gốc: giá trị của một nhánh phụ thuộc **hướng nhìn** (cha → con hay con → cha).
3. Viết DP xuống (`down`) và DP lên (`up`) với kỹ thuật **top-2 max / top-2 min**.
4. Viết 3 lời giải (thử mọi gốc, rerooting top-2, rerooting prefix/suffix) và đối chiếu bằng test ngẫu nhiên.
5. Phân tích `O(n)` và chú ý số lớn (giá trị có thể nhân đôi theo từng tầng).

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho `n` công việc `0..n-1` nối thành cây vô hướng bằng `edges` (độ dài `n − 1`), và `baseTime[i]` là thời gian cơ sở của công việc `i`.

Với một công việc được chọn làm **gốc**, thời điểm hoàn thành `finish(v)` tính như sau (con của `v` là các hàng xóm không phải cha của `v` trong cây có gốc đó):

- **Lá** (không có con): `finish(v) = baseTime[v]`.
- **Nút có con:** gọi `earliest` = min, `latest` = max của `finish` các con; `ownDuration = (latest − earliest) + baseTime[v]`; `finish(v) = latest + ownDuration = 2·latest − earliest + baseTime[v]`.

Chọn gốc tùy ý, tính `finish(gốc)`. Trả về `finish(gốc)` **nhỏ nhất** có thể qua mọi cách chọn gốc.

**Ví dụ 1:** `n = 3`, `edges = [[0,1],[1,2]]`, `baseTime = [9,1,5]` → `14` (gốc là `1`: hai lá `9`, `5`; `finish(1) = 9 + (9 − 5) + 1 = 14`).

**Ví dụ 2:** `n = 3`, `edges = [[0,1],[0,2]]`, `baseTime = [4,7,6]` → `12` (gốc là `0`).

**Ví dụ 3:** `n = 4`, `edges = [[0,1],[0,2],[2,3]]`, `baseTime = [5,8,2,1]` → `16` (gốc là `1`, đường `1 → 0 → 2 → 3`).

**Ràng buộc**

- `1 <= n <= 10^5`, `edges.length = n − 1`, `edges[i] = [u, v]`, `u != v`, đầu vào là cây hợp lệ
- `baseTime.length == n`, `1 <= baseTime[i] <= 10^5`

Lưu ý hai điểm dễ nhầm: (1) **gốc có đúng một con vẫn là nút trong** (không phải lá) — như gốc `1` ở ví dụ 3; (2) `n = 1` thì gốc duy nhất là lá nên đáp án là `baseTime[0]`.

In [1]:
examples = [
    (dict(n=3, edges=[[0, 1], [1, 2]], baseTime=[9, 1, 5]), 14),
    (dict(n=3, edges=[[0, 1], [0, 2]], baseTime=[4, 7, 6]), 12),
    (dict(n=4, edges=[[0, 1], [0, 2], [2, 3]], baseTime=[5, 8, 2, 1]), 16),
]
for args, expected in examples:
    print("n =", args["n"], "| edges =", args["edges"], "| baseTime =", args["baseTime"], "| mong đợi:", expected)

n = 3 | edges = [[0, 1], [1, 2]] | baseTime = [9, 1, 5] | mong đợi: 14
n = 3 | edges = [[0, 1], [0, 2]] | baseTime = [4, 7, 6] | mong đợi: 12
n = 4 | edges = [[0, 1], [0, 2], [2, 3]] | baseTime = [5, 8, 2, 1] | mong đợi: 16


## 3. Kiến thức nền

### 3.1. Cây vô hướng bằng danh sách kề

Mỗi cạnh `[u, v]` được thêm hai chiều: `adj[u].append(v)`, `adj[v].append(u)`.

In [2]:
n = 4
edges = [[0, 1], [0, 2], [2, 3]]
adj = [[] for _ in range(n)]
for u, v in edges:
    adj[u].append(v)
    adj[v].append(u)
print(adj)            # [[1, 2], [0], [0, 3], [2]]
print("bậc:", [len(a) for a in adj])

[[1, 2], [0], [0, 3], [2]]
bậc: [2, 1, 2, 1]


### 3.2. Cây có gốc: cha, con và thứ tự BFS

Khi chọn gốc `r`, ta tìm `parent` bằng BFS và ghi `order` (thứ tự thăm). Xử lý `reversed(order)` đảm bảo **con được xử lý trước cha** — thứ tự cần cho DP từ lá lên. Dùng vòng lặp thay vì đệ quy để tránh tràn ngăn xếp khi cây là một đường dài `10^5` nút.

In [3]:
def bfs_order(adj, root):
    parent = [-1] * len(adj)
    order = [root]
    for u in order:                      # vừa lặp vừa thêm vào cuối list
        for v in adj[u]:
            if v != parent[u]:
                parent[v] = u
                order.append(v)
    return parent, order

parent, order = bfs_order(adj, 0)
print("parent:", parent)
print("order :", order)
print("reversed (con trước cha):", order[::-1])

parent: [-1, 0, 0, 2]
order : [0, 1, 2, 3]
reversed (con trước cha): [3, 2, 1, 0]


### 3.3. "Con" phụ thuộc vào gốc: đổi gốc thì tập con đổi

Với cây `0 – 1 – 2` (đường thẳng): nếu gốc là `0` thì `1` có một con là `2`; nếu gốc là `2` thì `1` có con là `0`; nếu gốc là `1` thì `1` có hai con `0` và `2`. Tức "giá trị của nhánh" phụ thuộc **hướng nhìn**.

In [4]:
path = [[1], [0, 2], [1]]
for root in range(3):
    parent, order = bfs_order(path, root)
    children = {u: [v for v in path[u] if v != parent[u]] for u in range(3)}
    print("gốc", root, "-> con:", children)

gốc 0 -> con: {0: [1], 1: [2], 2: []}
gốc 1 -> con: {0: [], 1: [0, 2], 2: []}
gốc 2 -> con: {0: [], 1: [0], 2: [1]}


### 3.4. Giữ "top-2" để loại bỏ một phần tử khỏi min/max

Nếu có một nhóm giá trị và ta cần `max` / `min` của nhóm **sau khi bỏ một phần tử `x`**, chỉ cần biết hai giá trị lớn nhất (`m1 ≥ m2`) và hai giá trị nhỏ nhất (`s1 ≤ s2`):

- bỏ `x`: `max còn lại = m2` nếu `x == m1`, ngược lại `m1`;
- `min còn lại = s2` nếu `x == s1`, ngược lại `s1`.

Với giá trị lặp (ví dụ hai phần tử bằng nhau cùng là `m1`), điều này vẫn đúng vì `m2 == m1` khi đó.

In [5]:
def top2(vals):
    """trả về (max1, max2, min1, min2); max2/min2 = None nếu chỉ có 1 phần tử"""
    srt = sorted(vals)
    mn1 = srt[0]; mn2 = srt[1] if len(srt) > 1 else None
    mx1 = srt[-1]; mx2 = srt[-2] if len(srt) > 1 else None
    return mx1, mx2, mn1, mn2

vals = [5, 9, 9, 2]
mx1, mx2, mn1, mn2 = top2(vals)
print(top2(vals))
for x in vals:
    rest = list(vals); rest.remove(x)
    mx = mx2 if x == mx1 else mx1
    mn = mn2 if x == mn1 else mn1
    assert (max(rest), min(rest)) == (mx, mn)
print("loại bỏ bất kỳ phần tử nào bằng top-2: đúng")

(9, 9, 2, 5)
loại bỏ bất kỳ phần tử nào bằng top-2: đúng


## 4. Ý tưởng nền tảng / trực giác

**Tính tay ví dụ 3 với gốc `1`:** cây là `1 – 0 – 2 – 3` và `1` là gốc có đúng một con `0`; nút `0` có con `2`, nút `2` có con `3` (lá).

| Nút | Con | `earliest` | `latest` | `finish` |
|---|---|---|---|---|
| 3 | — (lá) | — | — | `baseTime[3] = 1` |
| 2 | `{3}` | 1 | 1 | `1 + (1 − 1) + 2 = 3` |
| 0 | `{2}` | 3 | 3 | `3 + 0 + 5 = 8` |
| 1 | `{0}` | 8 | 8 | `8 + 0 + 8 = 16` |

Đáp án khi gốc là `1` là `16` — trùng đáp án ví dụ. Với gốc `0` (hai con `1` và `2`): `finish(1) = 8` (lá), `finish(2) = 3`; `finish(0) = 2·8 − 3 + 5 = 18 > 16`.

**Quan sát:** nếu một nút chỉ có **một** con thì `earliest = latest` và `finish = finish(con) + baseTime`; nếu có hai con lệch nhau, nút bị phạt thêm `(latest − earliest)` — "đợi" nhánh chậm nhất và còn mất thêm chênh lệch.

Ta kiểm chứng bằng cách cài đặt trực tiếp định nghĩa đệ quy cho từng gốc.

In [6]:
def finish_literal(adj, base, v, parent):
    """Định nghĩa đệ quy, đọc đúng như đề (chỉ dùng cho cây nhỏ)."""
    children = [u for u in adj[v] if u != parent]
    if not children:
        return base[v]
    fins = [finish_literal(adj, base, u, v) for u in children]
    earliest, latest = min(fins), max(fins)
    own = (latest - earliest) + base[v]
    return latest + own

args = examples[2][0]
adj3 = [[] for _ in range(args["n"])]
for u, v in args["edges"]:
    adj3[u].append(v); adj3[v].append(u)
for root in range(4):
    print("gốc", root, "->", finish_literal(adj3, args["baseTime"], root, -1))

gốc 0 -> 18
gốc 1 -> 16
gốc 2 -> 27
gốc 3 -> 16


## 5. Cách 1 — Thử mọi gốc, mỗi gốc một lần DFS

**Ý tưởng.** Với mỗi `r` trong `0..n−1`, tính `finish(r)` bằng định nghĩa đệ quy (hoặc bằng BFS + xử lý ngược). Đáp án là `min` qua mọi `r`. Ở đây ta dùng bản vòng lặp (BFS) để tránh đệ quy sâu.

In [7]:
def build_adj(n, edges):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)
    return adj

def finish_from_root(adj, base, root):
    n = len(adj)
    parent, order = bfs_order(adj, root)
    f = [0] * n
    for u in reversed(order):
        ch = [f[v] for v in adj[u] if v != parent[u]]
        if not ch:
            f[u] = base[u]
        else:
            latest, earliest = max(ch), min(ch)
            f[u] = latest + (latest - earliest) + base[u]
    return f[root]

def try_all_roots(n, edges, baseTime):
    adj = build_adj(n, edges)
    return min(finish_from_root(adj, baseTime, r) for r in range(n))

for args, expected in examples:
    got = try_all_roots(**args)
    print(got, expected)
    assert got == expected

14 14
12 12
16 16


**Độ phức tạp.** Mỗi gốc `O(n)`, `n` gốc ⇒ `O(n^2)` thời gian, `O(n)` bộ nhớ. Với `n = 10^5` là `10^10` — quá chậm.

**Nút thắt.** Khi đổi gốc từ `u` sang một hàng xóm `v`, **gần như toàn bộ** cây giữ nguyên hướng; chỉ mấy nút trên đường từ gốc cũ tới gốc mới thay đổi hướng nhìn. Ta lặp lại rất nhiều công việc giống nhau. Cần tính trước giá trị của **mỗi nhánh theo mỗi hướng** — đó là rerooting.

## 6. Bước đột phá

**Insight 1 — một cạnh có hai giá trị.** Với mỗi cạnh `(p, v)`, có **hai** giá trị "nhánh":

- `down[v]` = `finish(v)` khi `v` là con của `p` (nhìn từ cha xuống) — phụ thuộc cây con của `v`;
- `up[v]` = `finish(p)` khi `p` là con của `v` (nhìn từ `v` lên phía cha) — phụ thuộc phần còn lại của cây khi bỏ cây con của `v`.

**Insight 2 — `finish` chỉ cần min, max của giá trị các nhánh.** `finish(v) = 2·max − min + baseTime[v]` trên tập các nhánh con. Nếu tập đó rỗng thì `finish(v) = baseTime[v]`.

**Insight 3 — giá trị của gốc.** Khi chọn nút `u` làm gốc, **mọi** hàng xóm là con của `u`, nên tập nhánh là `B(u) = { down[c] : c là con của u trong cây gốc 0 } ∪ { up[u] nếu u không phải gốc 0 }`. Đáp án ứng viên của `u` là `2·max(B(u)) − min(B(u)) + baseTime[u]` (hoặc `baseTime[u]` nếu `B(u)` rỗng, tức `n = 1`).

**Insight 4 — tính `up[c]` cho con `c` của `u`.** `up[c]` là `finish(u)` khi `u` coi mọi hàng xóm **trừ `c`** là con: tập nhánh là `B(u)` bỏ đi nhánh của `c` (giá trị `down[c]`). Dùng **top-2** để lấy `max`/`min` của `B(u) \ {down[c]}` trong `O(1)`.

**Insight 5 — thứ tự tính.** `down` tính từ lá lên (`reversed(order)`); `up` tính từ gốc xuống (`order`), vì `up[c]` cần `up[u]` của cha `u`.

**Quy tắc "nếu… thì…":**

- Nếu `u` không có nhánh nào ngoài `c` (tức `|B(u)| = 1`) thì `up[c] = baseTime[u]` (`u` thành lá khi nhìn từ `c`).
- Nếu `down[c] == max1` thì `max còn lại = max2`, ngược lại `max1` (tương tự `min`).
- `n = 1` ⇒ đáp án `baseTime[0]` (không có cạnh).

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Rerooting với top-2 max / top-2 min (bản chính, `O(n)`)

**Giải thích từng đoạn.**

1. Dựng `adj`, BFS từ `0` để có `parent`, `order`.
2. **DP xuống:** duyệt `reversed(order)`: với nút `u`, `down[u] = baseTime[u]` nếu không có con, ngược lại `2·max − min + baseTime[u]` trên `down` của các con.
3. **DP lên + đáp án:** duyệt `order`. Với nút `u`:
   - `vals` = `down[c]` cho mọi con `c` và thêm `up[u]` nếu `u` không phải gốc.
   - tính `max1, max2, min1, min2` của `vals` (một lần quét).
   - ứng viên đáp án: `2·max1 − min1 + baseTime[u]`.
   - với mỗi con `c`: `x = down[c]`; nếu `len(vals) == 1` thì `up[c] = baseTime[u]`; ngược lại `max còn lại`, `min còn lại` như trên và `up[c] = 2·maxEx − minEx + baseTime[u]`.
4. Đáp án = nhỏ nhất của các ứng viên.

In [8]:
def rerooting_top2(n, edges, baseTime):
    if n == 1:
        return baseTime[0]
    adj = build_adj(n, edges)
    parent, order = bfs_order(adj, 0)
    down = [0] * n
    for u in reversed(order):
        ch = [down[v] for v in adj[u] if v != parent[u]]
        down[u] = baseTime[u] if not ch else 2 * max(ch) - min(ch) + baseTime[u]
    up = [0] * n                                  # up[c]: finish(parent) khi nhìn từ c (không xác định với gốc 0)
    best = None
    for u in order:
        vals = [down[v] for v in adj[u] if v != parent[u]]
        if parent[u] != -1:
            vals.append(up[u])
        # top-2 max / top-2 min trong một lần quét
        mx1 = mx2 = None
        mn1 = mn2 = None
        for x in vals:
            if mx1 is None or x > mx1:
                mx2, mx1 = mx1, x
            elif mx2 is None or x > mx2:
                mx2 = x
            if mn1 is None or x < mn1:
                mn2, mn1 = mn1, x
            elif mn2 is None or x < mn2:
                mn2 = x
        cand = 2 * mx1 - mn1 + baseTime[u]
        if best is None or cand < best:
            best = cand
        for c in adj[u]:
            if c == parent[u]:
                continue
            if len(vals) == 1:
                up[c] = baseTime[u]
            else:
                x = down[c]
                mxe = mx2 if x == mx1 else mx1
                mne = mn2 if x == mn1 else mn1
                up[c] = 2 * mxe - mne + baseTime[u]
    return best

for args, expected in examples:
    got = rerooting_top2(**args)
    print(got, expected)
    assert got == expected

14 14
12 12
16 16


**Độ phức tạp.** Mỗi nút quét danh sách kề một lần: tổng `O(Σ deg) = O(n)`. Bộ nhớ `O(n)`.

**Kiểm tra nhanh `up`.** Với gốc 0, `up[c]` là `finish(parent)` khi nhìn từ `c`. Thử đối chiếu: `finish_from_root(adj, base, c)` có phải bằng `2·max − min + base[c]` trên `B(c)` hay không — ta sẽ làm trong phần test.

### 7.2. Cách 3 — Rerooting bằng prefix/suffix (tổng quát cho mọi phép gộp)

**Ý tưởng.** Top-2 là mẹo riêng cho `min/max`. Kỹ thuật **prefix/suffix** dùng được cho **bất kỳ** phép gộp kết hợp: với nút `u` có danh sách giá trị nhánh `vals[0..k−1]`, dựng:

- `pre[j]` = (min, max) của `vals[0..j−1]`,
- `suf[j]` = (min, max) của `vals[j..k−1]`.

Phép "bỏ phần tử `j`" = gộp `pre[j]` với `suf[j+1]`. Mẹo này đúng ngay cả khi các phần tử lặp lại (vì ta dựa vào **vị trí**, không dựa vào giá trị).

Ở đây phép gộp là `(min, max)` — hai đại lượng độc lập. Với bài khác (ví dụ tổng, tích, `gcd`) chỉ cần đổi phép gộp.

In [9]:
def rerooting_prefix_suffix(n, edges, baseTime):
    if n == 1:
        return baseTime[0]
    INF = float("inf")
    adj = build_adj(n, edges)
    parent, order = bfs_order(adj, 0)
    down = [0] * n
    for u in reversed(order):
        ch = [down[v] for v in adj[u] if v != parent[u]]
        down[u] = baseTime[u] if not ch else 2 * max(ch) - min(ch) + baseTime[u]
    up = [0] * n
    best = None

    def merge(a, b):                               # (min, max) của hai nhóm
        return (min(a[0], b[0]), max(a[1], b[1]))

    for u in order:
        kids = [c for c in adj[u] if c != parent[u]]
        vals = [down[c] for c in kids]
        if parent[u] != -1:
            vals.append(up[u])                     # nhánh phía cha đứng cuối
        k = len(vals)
        EMPTY = (INF, -INF)
        pre = [EMPTY] * (k + 1)
        suf = [EMPTY] * (k + 1)
        for j in range(k):
            pre[j + 1] = merge(pre[j], (vals[j], vals[j]))
        for j in range(k - 1, -1, -1):
            suf[j] = merge(suf[j + 1], (vals[j], vals[j]))
        mn, mx = pre[k]
        cand = 2 * mx - mn + baseTime[u]
        if best is None or cand < best:
            best = cand
        for j, c in enumerate(kids):               # vị trí j trong vals ứng với con c
            mn_ex, mx_ex = merge(pre[j], suf[j + 1])
            if mn_ex == INF:                       # không còn nhánh nào -> u là lá khi nhìn từ c
                up[c] = baseTime[u]
            else:
                up[c] = 2 * mx_ex - mn_ex + baseTime[u]
    return best

for args, expected in examples:
    got = rerooting_prefix_suffix(**args)
    print(got, expected)
    assert got == expected

14 14
12 12
16 16


**Độ phức tạp.** `O(n)` thời gian và bộ nhớ (mỗi nút dựng hai mảng độ dài `deg + 1`). Hằng số lớn hơn top-2 nhưng **tổng quát** hơn và ít sai sót với giá trị lặp.

## 8. TRACE từng bước

Ta trace `rerooting_top2` cho ví dụ 3: `edges = [[0,1],[0,2],[2,3]]`, `baseTime = [5,8,2,1]`. In `down`, rồi với từng nút: danh sách nhánh `vals`, top-2, ứng viên và `up` của các con.

Bảng mong đợi (gốc 0 để tính `down`):

| Nút | Con (gốc 0) | `down` |
|---|---|---|
| 3 | — | `1` |
| 2 | `{3}` | `1 + 0 + 2 = 3` |
| 1 | — | `8` |
| 0 | `{1, 2}` | `2·8 − 3 + 5 = 18` |

Sau đó `up`: ví dụ `up[1]` = `finish(0)` khi bỏ nhánh `1` = chỉ còn nhánh `2`: `3 + 0 + 5 = 8`. Ứng viên nút `1`: nhánh `{up[1] = 8}` → `8 + 0 + 8 = 16`.

In [10]:
def trace_reroot(n, edges, baseTime):
    adj = build_adj(n, edges)
    parent, order = bfs_order(adj, 0)
    down = [0] * n
    print("== DP xuống (từ lá lên), gốc 0 ==")
    for u in reversed(order):
        ch = [v for v in adj[u] if v != parent[u]]
        vals = [down[v] for v in ch]
        down[u] = baseTime[u] if not ch else 2 * max(vals) - min(vals) + baseTime[u]
        print(f"nút {u}: con={ch}, finish các con={vals} -> down[{u}] = {down[u]}")
    up = [None] * n
    best = None
    print("\n== DP lên + ứng viên mỗi gốc ==")
    for u in order:
        ch = [v for v in adj[u] if v != parent[u]]
        vals = [down[v] for v in ch] + ([up[u]] if parent[u] != -1 else [])
        srt = sorted(vals)
        mx1, mx2, mn1, mn2 = srt[-1], (srt[-2] if len(srt) > 1 else None), srt[0], (srt[1] if len(srt) > 1 else None)
        cand = 2 * mx1 - mn1 + baseTime[u]
        print(f"nút {u}: nhánh={vals}, max1={mx1}, max2={mx2}, min1={mn1}, min2={mn2} -> ứng viên gốc {u} = {cand}")
        best = cand if best is None else min(best, cand)
        for c in ch:
            if len(vals) == 1:
                up[c] = baseTime[u]
            else:
                x = down[c]
                mxe = mx2 if x == mx1 else mx1
                mne = mn2 if x == mn1 else mn1
                up[c] = 2 * mxe - mne + baseTime[u]
            print(f"    up[{c}] = (nhìn từ {c} lên {u}) = {up[c]}")
    print("\nđáp án =", best)
    return best

trace_reroot(**examples[2][0])

== DP xuống (từ lá lên), gốc 0 ==
nút 3: con=[], finish các con=[] -> down[3] = 1
nút 2: con=[3], finish các con=[1] -> down[2] = 3
nút 1: con=[], finish các con=[] -> down[1] = 8
nút 0: con=[1, 2], finish các con=[8, 3] -> down[0] = 18

== DP lên + ứng viên mỗi gốc ==
nút 0: nhánh=[8, 3], max1=8, max2=3, min1=3, min2=8 -> ứng viên gốc 0 = 18
    up[1] = (nhìn từ 1 lên 0) = 8
    up[2] = (nhìn từ 2 lên 0) = 13
nút 1: nhánh=[8], max1=8, max2=None, min1=8, min2=None -> ứng viên gốc 1 = 16
nút 2: nhánh=[1, 13], max1=13, max2=1, min1=1, min2=13 -> ứng viên gốc 2 = 27
    up[3] = (nhìn từ 3 lên 2) = 15
nút 3: nhánh=[15], max1=15, max2=None, min1=15, min2=None -> ứng viên gốc 3 = 16

đáp án = 16


16

In [11]:
# Đối chiếu từng gốc: ứng viên của rerooting phải bằng finish_from_root của đúng gốc đó
args = examples[2][0]
adj = build_adj(args["n"], args["edges"])
print("finish khi chọn từng nút làm gốc:", [finish_from_root(adj, args["baseTime"], r) for r in range(args["n"])])

finish khi chọn từng nút làm gốc: [18, 16, 27, 16]


Bạn có thể thấy danh sách `finish_from_root` cho từng gốc khớp với các "ứng viên" ở trace (ví dụ gốc `1` cho `16`, gốc `0` cho `18`). Điều đó chứng tỏ DP lên đúng: mỗi ứng viên là đáp án đúng cho gốc tương ứng, và ta lấy `min`.

## 9. Code nộp LeetCode

Dùng cách 2 (top-2). Dùng BFS thay vì đệ quy để an toàn với cây là một đường dài. Ghi chú: tên phương thức bài Premium là suy luận.

In [12]:
class Solution:
    def minFinishTime(self, n: int, edges: list[list[int]], baseTime: list[int]) -> int:
        if n == 1:
            return baseTime[0]
        adj = [[] for _ in range(n)]
        for u, v in edges:
            adj[u].append(v)
            adj[v].append(u)
        parent = [-1] * n
        order = [0]
        for u in order:                              # BFS từ gốc 0
            for v in adj[u]:
                if v != parent[u]:
                    parent[v] = u
                    order.append(v)

        down = [0] * n                               # finish(u) khi u là con của parent[u]
        for u in reversed(order):
            mx = mn = None
            for v in adj[u]:
                if v != parent[u]:
                    x = down[v]
                    if mx is None or x > mx: mx = x
                    if mn is None or x < mn: mn = x
            down[u] = baseTime[u] if mx is None else 2 * mx - mn + baseTime[u]

        up = [0] * n                                 # finish(parent[c]) khi nhìn từ c
        ans = None
        for u in order:
            vals = [down[v] for v in adj[u] if v != parent[u]]
            if parent[u] != -1:
                vals.append(up[u])
            mx1 = mx2 = mn1 = mn2 = None
            for x in vals:
                if mx1 is None or x > mx1:
                    mx2, mx1 = mx1, x
                elif mx2 is None or x > mx2:
                    mx2 = x
                if mn1 is None or x < mn1:
                    mn2, mn1 = mn1, x
                elif mn2 is None or x < mn2:
                    mn2 = x
            cand = 2 * mx1 - mn1 + baseTime[u]       # u làm gốc: mọi hàng xóm là con
            if ans is None or cand < ans:
                ans = cand
            for c in adj[u]:
                if c == parent[u]:
                    continue
                if len(vals) == 1:
                    up[c] = baseTime[u]
                else:
                    x = down[c]
                    up[c] = 2 * (mx2 if x == mx1 else mx1) - (mn2 if x == mn1 else mn1) + baseTime[u]
        return ans


sol = Solution()
for args, expected in examples:
    got = sol.minFinishTime(**args)
    print(got, expected)
    assert got == expected

14 14
12 12
16 16


## 10. Vì sao thuật toán đúng

**Đệ quy chỉ phụ thuộc cây con.** Với gốc `r`, `finish(v)` chỉ phụ thuộc cây con của `v` (các hàng xóm trừ cha). Cây con của `v` chỉ phụ thuộc **hướng** từ cha tới `v`. Vì vậy với mỗi cạnh có hướng `(p → v)` giá trị `F(p → v) = finish(v)` khi cha là `p` là **cố định**, không phụ thuộc gốc nằm ở đâu phía sau `p`.

**Down.** `down[v] = F(parent[v] → v)` cho cây gốc 0 — tính từ lá lên theo định nghĩa.

**Up.** `up[c] = F(c → u)` với `u = parent[c]`: nút `u` có hàng xóm là các con khác của `u` (mỗi con `c'` cho nhánh `F(u → c') = down[c']`) và cha của `u` (nhánh `F(u → parent[u]) = up[u]`). `F(c → u)` là `finish(u)` khi **loại `c`** khỏi tập con ⇒ `2·max − min + base` trên `B(u)` bỏ nhánh `down[c]`; nếu tập đó rỗng thì `u` là lá nên `base[u]`. Đúng theo quy nạp theo độ sâu (duyệt theo `order`, `up[u]` có trước khi tính `up[c]`).

**Ứng viên gốc.** Khi chọn `u` làm gốc, các con của `u` là **tất cả** hàng xóm; giá trị mỗi nhánh là `down[c]` hoặc `up[u]` (đã đúng như trên). Ứng viên bằng `2·max − min + base[u]` (nếu `n ≥ 2` thì `u` có ít nhất một hàng xóm nên không phải lá). Lấy `min` qua mọi `u` là đáp án.

**Top-2 đúng khi có giá trị lặp.** Giả sử các giá trị `{9, 9, 2}`. `max1 = max2 = 9`. Bỏ một `9`: còn `{9, 2}` có `max = 9 = max2`. Công thức `x == mx1 ? mx2 : mx1` cho `mx2 = 9` ✓. Bỏ `2`: `x ≠ mx1` ⇒ `max = mx1 = 9` ✓.

**Khớp với định nghĩa lá.** Một nút không phải gốc có bậc `1` là lá (`down = base`); trong DP lên, `|B(u)| = 1` nghĩa là sau khi bỏ `c` không còn nhánh ⇒ `u` là lá khi nhìn từ `c` ⇒ `up[c] = base[u]`.

## 11. Phân tích Time/Space

Đặt `n` = số công việc.

| Cách | Time | Space |
|---|---|---|
| Thử mọi gốc | `O(n^2)` | `O(n)` |
| Rerooting top-2 (nộp) | `O(n)` | `O(n)` |
| Rerooting prefix/suffix | `O(n)` | `O(n)` |

Lưu ý: giá trị `finish` có thể nhân đôi theo từng tầng (khi con nặng nhất vượt xa con nhẹ nhất) nên về lý thuyết có thể rất lớn; Python dùng số nguyên lớn nên không tràn, và test của đề được kiểm soát để đáp án vừa trong giới hạn của ngôn ngữ khác.

In [13]:
import time, random
rng = random.Random(5)
n = 100000
# (a) cây ngẫu nhiên, nhãn xáo trộn
labels = list(range(n)); rng.shuffle(labels)
edges = [[labels[i], labels[rng.randrange(i)]] for i in range(1, n)]
base = [rng.randint(1, 10**5) for _ in range(n)]
t0 = time.time(); r1 = Solution().minFinishTime(n, edges, base); print("cây ngẫu nhiên n=1e5:", r1, f"({time.time()-t0:.2f}s)")
# (b) đường thẳng dài (đệ quy sẽ tràn ngăn xếp; BFS thì không)
edges = [[i, i + 1] for i in range(n - 1)]
t0 = time.time(); r2 = Solution().minFinishTime(n, edges, base); print("đường thẳng n=1e5   :", r2, f"({time.time()-t0:.2f}s)")

cây ngẫu nhiên n=1e5: 128255413292 (0.27s)
đường thẳng n=1e5   : 2500476828 (0.10s)


## 12. Bộ test

Chiến lược test:

1. 3 ví dụ của đề (assert).
2. Biên: `n = 1`; `n = 2`; đường thẳng; ngôi sao (một nút nối mọi nút khác); hai nhánh bằng nhau (giá trị lặp ở top-2).
3. Phản ví dụ: coi gốc có một con là lá (phần 13).
4. Đối chiếu định nghĩa đệ quy trực tiếp (`finish_literal`), thử mọi gốc, hai bản rerooting và `Solution` trên cây ngẫu nhiên (seed cố định), kể cả cây "sâu" (caterpillar) để thử số lớn.

In [14]:
sol = Solution()
for args, expected in examples:
    assert sol.minFinishTime(**args) == expected

assert sol.minFinishTime(1, [], [7]) == 7
assert sol.minFinishTime(2, [[0, 1]], [3, 4]) == min(3 + 0 + 4, 4 + 0 + 3) == 7   # gốc có một con
# ngôi sao: tâm 0, lá 1..3 với base 10,20,30
assert sol.minFinishTime(4, [[0, 1], [0, 2], [0, 3]], [1, 10, 20, 30]) == 51
# giá trị lặp: hai lá bằng nhau
assert sol.minFinishTime(3, [[0, 1], [0, 2]], [5, 9, 9]) == min(2 * 9 - 9 + 5, 9 + 5 + 0 + 9)
print("test cố định OK")

test cố định OK


In [15]:
def random_tree_edges(rng, n):
    labels = list(range(n)); rng.shuffle(labels)
    return [[labels[i], labels[rng.randrange(i)]] for i in range(1, n)]

def brute_force(n, edges, baseTime):
    adj = build_adj(n, edges)
    return min(finish_literal(adj, baseTime, r, -1) for r in range(n))

rng = random.Random(24680)
cnt = 0
for _ in range(1500):
    n = rng.randint(1, 9)
    edges = random_tree_edges(rng, n)
    base = [rng.randint(1, 20) for _ in range(n)]
    args = dict(n=n, edges=edges, baseTime=base)
    expected = brute_force(**args)
    assert try_all_roots(**args) == expected, args
    assert rerooting_top2(**args) == expected, args
    assert rerooting_prefix_suffix(**args) == expected, args
    assert Solution().minFinishTime(**args) == expected, args
    cnt += 1
print("đã đối chiếu", cnt, "cây ngẫu nhiên nhỏ: tất cả khớp định nghĩa đệ quy")

đã đối chiếu 1500 cây ngẫu nhiên nhỏ: tất cả khớp định nghĩa đệ quy


In [16]:
# Cây vừa, và cây "caterpillar" (xương sống + lá) để thử giá trị tăng nhanh
rng = random.Random(97531)
for _ in range(40):
    n = rng.randint(20, 120)
    edges = random_tree_edges(rng, n)
    base = [rng.randint(1, 10**5) for _ in range(n)]
    args = dict(n=n, edges=edges, baseTime=base)
    e = try_all_roots(**args)
    assert rerooting_top2(**args) == e == rerooting_prefix_suffix(**args) == Solution().minFinishTime(**args)

spine = 150                      # xương sống 0..149, mỗi nút xương sống có thêm 1 lá
edges = [[i, i + 1] for i in range(spine - 1)] + [[i, spine + i] for i in range(spine)]
n = 2 * spine
base = [rng.randint(1, 10**5) for _ in range(n)]
args = dict(n=n, edges=edges, baseTime=base)
e = try_all_roots(**args)
assert Solution().minFinishTime(**args) == e == rerooting_prefix_suffix(**args)
print("cây vừa & caterpillar khớp; giá trị đáp án caterpillar có", e.bit_length(), "bit")

cây vừa & caterpillar khớp; giá trị đáp án caterpillar có 92 bit


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Coi gốc có đúng một con là lá

**Code sai:** `if len(adj[u]) == 1: return baseTime[u]` áp dụng cả cho gốc.
**Vì sao sai:** đề định nghĩa lá là nút **không có con**. Gốc bậc 1 vẫn có một con ⇒ phải áp dụng công thức nút trong. Ví dụ 3: gốc `1` cho `16`, không phải `8`.
**Code đúng:** chỉ nút **không phải gốc** có bậc 1 mới là lá; với gốc phải xét tập con.

In [17]:
def wrong_degree_leaf(n, edges, baseTime):
    adj = build_adj(n, edges)
    best = None
    for root in range(n):
        parent, order = bfs_order(adj, root)
        f = [0] * n
        for u in reversed(order):
            if len(adj[u]) == 1:                      # SAI: kể cả gốc bậc 1
                f[u] = baseTime[u]
                continue
            ch = [f[v] for v in adj[u] if v != parent[u]]
            f[u] = 2 * max(ch) - min(ch) + baseTime[u]
        best = f[root] if best is None else min(best, f[root])
    return best

print("sai :", wrong_degree_leaf(**examples[2][0]), " đúng:", Solution().minFinishTime(**examples[2][0]))

sai : 1  đúng: 16


### Lỗi 2 — Chỉ tính một chiều (gốc cố định 0) rồi tưởng đã xong

**Code sai:** trả về `down[0]`.
**Vì sao sai:** đề hỏi `min` qua **mọi** gốc; gốc 0 chưa chắc tốt nhất (ví dụ 1: gốc 1 cho `14`, gốc 0 cho `15`).
**Code đúng:** thêm DP lên và lấy `min` các ứng viên.

In [18]:
def wrong_only_root0(n, edges, baseTime):
    adj = build_adj(n, edges)
    parent, order = bfs_order(adj, 0)
    return finish_from_root(adj, baseTime, 0)         # SAI: chỉ gốc 0

print("sai :", wrong_only_root0(**examples[0][0]), " đúng:", Solution().minFinishTime(**examples[0][0]))

sai : 15  đúng: 14


### Lỗi 3 — Loại bỏ phần tử khỏi min/max sai khi có giá trị lặp

**Code sai:** dùng "max còn lại = max1 nếu `x != max1`, ngược lại `max2`" nhưng khởi tạo `max2` bằng giá trị **khác** `max1` (loại trùng) hoặc dùng `set`.
**Vì sao sai:** với `{9, 9, 2}`, bỏ một `9` còn `max = 9`; nếu `max2` bị khử trùng thành `2` ta ra `2`.
**Code đúng:** giữ top-2 **có lặp** (`x > mx2` cập nhật cả khi `x == mx1`: nhánh `elif mx2 is None or x > mx2` nhận `x = mx1` thành `mx2 = mx1`).

In [19]:
def wrong_dedup_top2(n, edges, baseTime):
    # dùng set để lấy top-2: SAI khi có giá trị lặp
    adj = build_adj(n, edges)
    parent, order = bfs_order(adj, 0)
    down = [0] * n
    for u in reversed(order):
        ch = [down[v] for v in adj[u] if v != parent[u]]
        down[u] = baseTime[u] if not ch else 2 * max(ch) - min(ch) + baseTime[u]
    up = [0] * n
    best = None
    for u in order:
        vals = [down[v] for v in adj[u] if v != parent[u]] + ([up[u]] if parent[u] != -1 else [])
        s = sorted(set(vals))                          # SAI: mất bản sao
        mx1, mx2 = s[-1], (s[-2] if len(s) > 1 else None)
        mn1, mn2 = s[0], (s[1] if len(s) > 1 else None)
        cand = 2 * mx1 - mn1 + baseTime[u]
        best = cand if best is None else min(best, cand)
        for c in adj[u]:
            if c == parent[u]:
                continue
            if len(vals) == 1:
                up[c] = baseTime[u]
            else:
                x = down[c]
                mxe = mx2 if x == mx1 and mx2 is not None else mx1
                mne = mn2 if x == mn1 and mn2 is not None else mn1
                up[c] = 2 * mxe - mne + baseTime[u]
    return best

rng = random.Random(6)
for _ in range(3000):
    n = rng.randint(3, 8)
    edges = random_tree_edges(rng, n)
    base = [rng.randint(1, 3) for _ in range(n)]      # giá trị nhỏ -> nhiều lặp
    args = dict(n=n, edges=edges, baseTime=base)
    if wrong_dedup_top2(**args) != brute_force(**args):
        print("phản ví dụ:", args, "sai:", wrong_dedup_top2(**args), "đúng:", brute_force(**args))
        break

phản ví dụ: {'n': 6, 'edges': [[5, 3], [0, 3], [2, 3], [4, 2], [1, 3]], 'baseTime': [1, 3, 1, 1, 3, 1]} sai: 7 đúng: 8


### Lỗi 4 — Dùng đệ quy cho cây đường thẳng `10^5` nút

**Code sai:** `def dfs(u, p): ... dfs(v, u)` mặc định giới hạn đệ quy Python ~1000.
**Vì sao sai:** `RecursionError` hoặc tràn bộ nhớ ngăn xếp.
**Code đúng:** BFS + xử lý `reversed(order)` như ở các cách trên (không đệ quy).

## 14. Mẹo & nhận dạng pattern

- **"Chọn gốc tốt nhất của cây", "tính đáp án cho mọi gốc"** → rerooting: `down` (cây con), `up` (phần còn lại), rồi gộp.
- Nếu công thức nút chỉ phụ thuộc `min/max/tổng` của các con → dùng **top-2 / tổng trừ đi phần tử** để loại một con trong `O(1)`; nếu phép gộp phức tạp thì **prefix/suffix**.
- Phân biệt "lá" (không có con) với "nút bậc 1" khi nút đó là gốc.
- Luôn dùng BFS/`order` đảo ngược thay vì đệ quy khi `n = 10^5` và cây có thể dài như đường thẳng.
- Kiểm chứng rerooting bằng cách so với "thử mọi gốc" trên cây nhỏ: đây là phép thử rất mạnh.
- Cẩn thận số lớn khi công thức có hệ số `2·latest` (nhân đôi theo từng tầng).

## 15. Biến thể

1. **Tìm gốc cho `finish` lớn nhất** (Bài 5) — đổi `min` thành `max`.
2. **Chỉ cho phép gốc là một lá:** lọc các nút có bậc 1 khi lấy `min` ứng viên.
3. **Trả về cả gốc tốt nhất** (chỉ số nhỏ nhất nếu hòa): theo dõi `argmin`.
4. **Có thêm hệ số cho nút:** `finish = latest + k·(latest − earliest) + base` với `k` tuỳ ý: công thức dùng `max`/`min` vẫn giống nhau; đổi `up/down` tương ứng.
5. **Đồ thị có chu trình (không phải cây):** bài khác hẳn; rerooting không dùng được.

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi gốc | mỗi gốc một lần DFS/BFS | `O(n^2)` | `O(n)` | đối chiếu `n` nhỏ |
| 2. Rerooting top-2 | `down`, `up` với top-2 min/max | `O(n)` | `O(n)` | **nộp bài** |
| 3. Rerooting prefix/suffix | gộp tiền tố và hậu tố theo vị trí | `O(n)` | `O(n)` | phép gộp tổng quát, tránh lỗi giá trị lặp |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** Với cây đường thẳng `0 – 1 – 2` (ví dụ 1), `baseTime = [9, 1, 5]`, `finish_from_root(adj, base, 0)` bằng bao nhiêu?

**Bài 2 (tính tay).** Ngôi sao tâm `0` và lá `1, 2, 3`, `baseTime = [1, 10, 20, 30]`. Tính `finish` với gốc là `0` và gốc là `2`; đáp án bài toán là gì?

**Bài 3 (điền TODO).** Hoàn thiện `top2_max_min(vals)` trả về `(max1, max2, min1, min2)` (giữ giá trị lặp; `None` nếu thiếu).

**Bài 4 (tự viết lại).** Viết `my_reroot(n, edges, baseTime)` theo cách 2 mà không nhìn code nộp.

**Bài 5 (biến thể).** Viết `max_finish_time(n, edges, baseTime)`: **lớn nhất** `finish(gốc)` qua mọi gốc.

In [20]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def top2_max_min(vals):
    # TODO: một lần quét, giữ max1 >= max2 và min1 <= min2 (có lặp); None nếu không đủ phần tử
    pass

# Bài 4 — TODO
def my_reroot(n, edges, baseTime):
    # TODO 1: n == 1 -> baseTime[0]; dựng adj, parent, order (BFS)
    # TODO 2: down[] theo reversed(order)
    # TODO 3: duyệt order: vals, top-2, ứng viên, up[c] cho từng con
    pass

# Bài 5 — TODO
def max_finish_time(n, edges, baseTime):
    # TODO: như my_reroot nhưng lấy max của các ứng viên
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: gốc `0`, đường `0 → 1 → 2`.
- Mức 2: `finish(2) = 5` (lá); `finish(1)` có một con: `5 + 0 + 1 = 6`.
- Mức 3: `finish(0) = 6 + 0 + 9 = 15`.

**Bài 2.**
- Mức 1: gốc `0`: ba con là lá `10, 20, 30`.
- Mức 2: `earliest = 10`, `latest = 30` ⇒ `2·30 − 10 + 1 = 51`. Gốc `2`: con duy nhất `0` có hai con `1, 3` (lá `10, 30`) ⇒ `finish(0) = 2·30 − 10 + 1 = 51`; gốc `2` một con ⇒ `51 + 0 + 20 = 71`.
- Mức 3: gốc `1`: `finish(0) = 2·30 − 20 + 1 = 41` (hai con của `0` là lá `20, 30`), nên `41 + 10 = 51`; gốc `3`: `finish(0) = 2·20 − 10 + 1 = 31`, nên `31 + 30 = 61`. Nhỏ nhất là `51`.

**Bài 3.**
- Mức 1: khởi tạo bốn biến `None`.
- Mức 2: với mỗi `x` cập nhật `max1/max2` bằng so sánh `x > max1` rồi `elif max2 is None or x > max2`.
- Mức 3: tương tự cho `min`.

**Bài 4.**
- Mức 1: giống cách 2.
- Mức 2: `up[c] = base[u]` khi `u` chỉ có một nhánh.
- Mức 3: `x == max1` thì dùng `max2`.

**Bài 5.**
- Mức 1: giữ nguyên toàn bộ DP.
- Mức 2: chỉ khác cách gộp ứng viên.
- Mức 3: `max` thay `min`.

## 19. Đáp án tham khảo

In [21]:
# Bài 1
adj_path = build_adj(3, [[0, 1], [1, 2]])
assert finish_from_root(adj_path, [9, 1, 5], 0) == 15
# Bài 2
adj_star = build_adj(4, [[0, 1], [0, 2], [0, 3]])
base_star = [1, 10, 20, 30]
assert [finish_from_root(adj_star, base_star, r) for r in range(4)] == [51, 51, 71, 61]
assert Solution().minFinishTime(4, [[0, 1], [0, 2], [0, 3]], base_star) == 51

# Bài 3
def top2_max_min_ans(vals):
    mx1 = mx2 = mn1 = mn2 = None
    for x in vals:
        if mx1 is None or x > mx1:
            mx2, mx1 = mx1, x
        elif mx2 is None or x > mx2:
            mx2 = x
        if mn1 is None or x < mn1:
            mn2, mn1 = mn1, x
        elif mn2 is None or x < mn2:
            mn2 = x
    return mx1, mx2, mn1, mn2

for vals in [[5], [5, 5], [9, 9, 2], [1, 2, 3, 4], [4, 3, 2, 1], [7, 1, 7, 1]]:
    s = sorted(vals)
    exp = (s[-1], s[-2] if len(s) > 1 else None, s[0], s[1] if len(s) > 1 else None)
    assert top2_max_min_ans(vals) == exp, vals
print("Bài 1–3 OK")

Bài 1–3 OK


In [22]:
# Bài 4 + Bài 5 (gộp một hàm trả về mọi ứng viên)
def all_candidates(n, edges, baseTime):
    if n == 1:
        return [baseTime[0]]
    adj = build_adj(n, edges)
    parent, order = bfs_order(adj, 0)
    down = [0] * n
    for u in reversed(order):
        ch = [down[v] for v in adj[u] if v != parent[u]]
        down[u] = baseTime[u] if not ch else 2 * max(ch) - min(ch) + baseTime[u]
    up = [0] * n
    cands = [None] * n
    for u in order:
        vals = [down[v] for v in adj[u] if v != parent[u]] + ([up[u]] if parent[u] != -1 else [])
        mx1, mx2, mn1, mn2 = top2_max_min_ans(vals)
        cands[u] = 2 * mx1 - mn1 + baseTime[u]
        for c in adj[u]:
            if c == parent[u]:
                continue
            if len(vals) == 1:
                up[c] = baseTime[u]
            else:
                x = down[c]
                up[c] = 2 * (mx2 if x == mx1 else mx1) - (mn2 if x == mn1 else mn1) + baseTime[u]
    return cands

def my_reroot_ans(n, edges, baseTime):
    return min(all_candidates(n, edges, baseTime))

def max_finish_time_ans(n, edges, baseTime):
    return max(all_candidates(n, edges, baseTime))

rng = random.Random(8)
for _ in range(500):
    n = rng.randint(1, 9)
    edges = random_tree_edges(rng, n)
    base = [rng.randint(1, 15) for _ in range(n)]
    adj = build_adj(n, edges)
    per_root = [finish_literal(adj, base, r, -1) for r in range(n)]
    assert my_reroot_ans(n, edges, base) == min(per_root) == Solution().minFinishTime(n, edges, base)
    assert max_finish_time_ans(n, edges, base) == max(per_root)
    assert sorted(all_candidates(n, edges, base)) == sorted(per_root)      # mỗi gốc đúng một ứng viên
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template rerooting với top-2 min/max:**

```python
parent, order = BFS từ 0
for u in reversed(order):                      # down: cây con của u (u là con của parent[u])
    ch = [down[v] for v in adj[u] if v != parent[u]]
    down[u] = base[u] if not ch else 2*max(ch) - min(ch) + base[u]
for u in order:                                # up + đáp án
    vals = [down[v] for v in children(u)] + ([up[u]] if u != gốc else [])
    mx1, mx2, mn1, mn2 = top-2 có lặp của vals
    ans = min(ans, 2*mx1 - mn1 + base[u])      # u làm gốc: mọi hàng xóm là con
    for c in children(u):
        up[c] = base[u] if len(vals) == 1 else \
                2*(mx2 if down[c]==mx1 else mx1) - (mn2 if down[c]==mn1 else mn1) + base[u]
```

**Checklist trước khi submit:**

- [ ] `n = 1` trả `baseTime[0]`.
- [ ] Lá = nút **không có con** (gốc bậc 1 không phải lá).
- [ ] `finish = 2·latest − earliest + base` khi có con; `base` khi không có con.
- [ ] `down` tính từ lá lên (`reversed(order)`), `up` từ gốc xuống (`order`).
- [ ] Top-2 giữ giá trị lặp; loại bỏ phần tử bằng cách so sánh với `max1`/`min1`.
- [ ] Trường hợp `|vals| == 1` ⇒ `up[c] = base[u]`.
- [ ] Không dùng đệ quy cho cây sâu (`n = 10^5`).
- [ ] Tên phương thức bài Premium cần đối chiếu lại với trình soạn thảo thật.
- [ ] Đã đối chiếu "thử mọi gốc" trên cây nhỏ ngẫu nhiên.